# FightWatch — pilot mở rộng, không dùng NTU

Mục tiêu demo: nhóm đông trong lớp học, hành lang và ngoài trời, camera iPhone cầm tay. Dữ liệu hiện có là UBI-Fights; **không coi đây là dữ liệu đã xác minh trùng môi trường demo**. Không cần tài khoản NTU.

Chọn 200 video nguồn từ official train, lấy tối đa 24 cửa sổ/video (mục tiêu vài nghìn clip), không chồng cửa sổ. Giữ validation cũ ngoài train; nhóm cảnh gần trùng xuyên split bị cách ly. Không loại mẫu đánh nhau chỉ vì YOLO đếm thiếu người. Mẫu âm ưu tiên có người, nền không người tối đa 5% train sau lọc. Mô phỏng rung máy nhẹ cho cả hai lớp trong fine-tune, không thay thế kiểm thử điện thoại thực tế.

Code sửa ở local rồi push GitHub. Cell 1 chủ động bỏ thay đổi **tracked** ở bản repo trên Drive khi đồng bộ; không xóa dataset/artifact. Checkpoint chưa được duyệt demo. Chỉ chạy từng cell; không train nhiều giờ tự động.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, shutil, json, time, threading
ROOT = Path('/content/drive/MyDrive/SENTINEL')
REPO = ROOT / 'repo/anomaly-behavior-detector-system'
LOCAL = Path('/content/fightwatch-expand-code')
SOURCE = ROOT / 'artifacts/fight_pilot_x3d_v1'
OUTPUT = ROOT / 'artifacts/fight_expand_v1'
DATA = Path('/content/fight_expand_data/UBI_FIGHTS')
CACHE = Path('/content/fight_expand_cache')
ARCHIVE = ROOT / 'datasets/fight/UBI_FIGHTS.zip'
BASELINE = ROOT / 'artifacts/fight_ablation_v2/interaction/runtime/models/fight_model.ts'
URL = 'https://github.com/TranDuon/anomaly-behavior-detector-system.git'
BUDGET_MINUTES = 45
for directory in (REPO.parent, LOCAL, OUTPUT, DATA, CACHE):
    directory.mkdir(parents=True, exist_ok=True)
assert ARCHIVE.exists() and ARCHIVE.stat().st_size == 8_117_112_419, 'Thiếu archive UBI gốc hoặc kích thước không khớp'
assert (SOURCE / 'manifest.json').exists() and BASELINE.exists(), 'Thiếu manifest/baseline đã lưu'
if (REPO / '.git').exists():
    remote = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    assert remote.removesuffix('.git') == URL.removesuffix('.git'), 'Sai remote'
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', URL, str(REPO)], check=True)
tracked = subprocess.check_output(['git', '-C', str(REPO), 'ls-files', '-z']).split(b'\0')
for name in tracked:
    if not name:
        continue
    relative = Path(os.fsdecode(name))
    if relative.name == '.env':
        continue
    destination = LOCAL / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / relative, destination)
os.chdir(LOCAL)
revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('REVISION', revision)
print('OUTPUT', OUTPUT)


## 2. GPU và kiểm thử
Chọn GPU L4 trong Runtime → Change runtime type trước cell này. Bộ đếm 45 phút tính từ cell này, gồm cài thư viện, chuẩn bị dữ liệu và tiến trình train. Watchdog dừng tiến trình khi hết giờ; không tự mở lại hoặc tự xóa runtime. Ngắt kết nối runtime sau khi xem/lưu kết quả để tránh tiếp tục giữ GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Cần chọn GPU trước khi tiếp tục'
PILOT_STARTED = time.monotonic()
PILOT_DEADLINE = PILOT_STARTED + BUDGET_MINUTES * 60
print('GPU', torch.cuda.get_device_name(0))
from scripts.colab_process import run_logged as _run_logged
def run_logged(command, timeout, name):
    return _run_logged(command, OUTPUT / name, timeout, revision)
def remaining():
    return max(0, PILOT_DEADLINE - time.monotonic())
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], min(480, remaining()), 'setup_log.txt')
run_logged([sys.executable, '-m', 'pytest', 'tests/test_fight_expand.py', 'tests/test_fight_ablation.py', 'tests/test_fight_runtime.py', '-q', '-p', 'no:cacheprovider'], min(180, remaining()), 'setup_log.txt')
(OUTPUT / 'code_revision.txt').write_text(revision)


## 3. Chuẩn bị đúng subset, không giải nén toàn bộ 80 giờ
Đọc archive đã có trên Drive, chỉ chép video được chọn sang `/content`. Manifest được đóng băng; nếu tham số khác lần trước, dùng thư mục output mới. Giới hạn tối đa 15 phút cho bước này. Đây vẫn là development validation, không phải test độc lập.

In [ ]:
common = ['--source-run', str(SOURCE), '--dataset-root', str(DATA), '--output-dir', str(OUTPUT)]
command = [sys.executable, '-u', '-m', 'scripts.fight_expand', 'prepare', *common, '--archive', str(ARCHIVE), '--exclude', str(ROOT / 'artifacts/fight/dedup/exclude.json'), '--videos', '200', '--clips-per-video', '24']
run_logged(command, min(900, remaining()), 'prepare_log.txt')
print((OUTPUT / 'data_summary.json').read_text())


## 4. Cache → baseline đối chứng → probe → fine-tune → đánh giá → gói CPU
Trim viền đen, nối bbox qua ba frame để giữ vùng chuyển động thay vì đếm lặp một người. Train/runtime dùng chung hàm tạo clip. Baseline cũ được đo lại trên **cùng tập validation và cùng cửa sổ thời gian**; không so trực tiếp headline với pilot nhỏ trước.

Cache clip RGB lớn nằm trên `/content` để đọc nhanh. Drive giữ manifest, bảng đặc trưng nén, log từng epoch, best checkpoint, dự đoán và gói runtime. Nếu hết budget giữa epoch, dùng best checkpoint của epoch hoàn tất để đánh giá; nếu chưa có epoch hoàn tất, ghi lỗi inconclusive. Không tự tăng thời gian.

In [ ]:
if (OUTPUT / 'report.json').exists():
    print('Đã có report, không train lại:', OUTPUT / 'report.json')
else:
    minutes = (remaining() - 30) / 60
    assert minutes > 5, 'Không đủ budget: dừng, không tự cấp thêm GPU'
    command = [sys.executable, '-u', '-m', 'scripts.fight_expand', 'train', *common, '--cache-dir', str(CACHE), '--baseline-model', str(BASELINE), '--budget-minutes', str(minutes)]
    run_logged(command, remaining(), 'run_log.txt')
print('Elapsed minutes:', round((time.monotonic() - PILOT_STARTED) / 60, 2))


## 5. Đọc kết quả và xuất clip lỗi
P/R/F1 phụ thuộc threshold; AP, AUC và in-video AUC giúp kiểm tra model có nhận chuyển động trong cùng cảnh không. Recall@precision75 là chỉ số thăm dò trên development validation, không đảm bảo precision75 ở camera thật. `approved_for_demo` luôn False ở pilot.

Audit chỉ cắt tối đa 6 báo nhầm + 6 bỏ sót, giữ cả video nguồn và crop để xem mất hành động hay model hiểu sai. Chỉ chạy khi còn cache/dataset trong phiên; không cần train lại.

In [ ]:
import pandas as pd
from IPython.display import display, Video
report = json.loads((OUTPUT / 'report.json').read_text())
summary = []
for name, values in [('v2_interaction_baseline', report['baseline_on_same_validation']), ('expanded', report)]:
    summary.append({'model': name, **values['validation'], 'recall_at_precision75': values['best_recall_at_precision75']})
table = pd.DataFrame(summary).set_index('model')
display(table)
(OUTPUT / 'summary.html').write_text(table.to_html())
print('Decision:', report['decision'], '| Epochs:', report['completed_epochs'])
print('Budget exhausted:', report['training_budget_exhausted'])
print('Train:', report['train_clips'], '| Validation:', report['validation_clips'], '| Videos:', report['validation_videos'])
print('Demo approved:', report['approved_for_demo'])
if not (OUTPUT / 'errors.json').exists() and CACHE.exists():
    run_logged([sys.executable, '-u', '-m', 'scripts.fight_expand', 'audit', *common, '--cache-dir', str(CACHE)], 180, 'audit_log.txt')
if (OUTPUT / 'errors.json').exists():
    errors = json.loads((OUTPUT / 'errors.json').read_text())
    for kind in ('false_positive', 'false_negative'):
        entry = next((entry for entry in errors if entry['kind'] == kind and 'source_clip' in entry), None)
        if entry:
            print(kind, entry['video'], entry['start_seconds'], 'score=', entry['score'])
            display(Video(str(OUTPUT / entry['source_clip']), embed=True, width=480))
            display(Video(str(OUTPUT / entry['crop_clip']), embed=True, width=480))
print('Saved:', OUTPUT)


## Kiểm tra model có dùng thông tin thời gian không (không train)
Lặp frame giữa và xáo thứ tự frame trên cùng validation, giữ nguyên threshold. Hai can thiệp là dữ liệu ngoài phân phối; điểm giống nhau gợi ý model thiên về ngoại hình/cảnh, không đủ để chứng minh nguyên nhân. Chỉ chạy khi còn budget/cache; không tự tăng thời gian.

In [ ]:
if (OUTPUT / 'temporal_controls.json').exists():
    print((OUTPUT / 'temporal_controls.json').read_text())
elif remaining() > 30:
    run_logged([sys.executable, '-u', '-m', 'scripts.fight_controls', '--output-dir', str(OUTPUT), '--cache-dir', str(CACHE), '--budget-seconds', str(min(150, remaining() - 10))], min(180, remaining()), 'temporal_audit_log.txt')
else:
    print('Không đủ budget cho audit thời gian, không tự cấp thêm GPU.')


## 6. Tải gói CPU về máy (tùy chọn)
Chỉ tải nếu muốn thử checkpoint pilot; không ghi đè model local một cách âm thầm. Để ZIP trong Downloads rồi chạy `run_local.cmd`; phải chủ động bật cho phép checkpoint pilot trong UI. Crop mode mới cần code local cùng revision hoặc mới hơn.

In [ ]:
from google.colab import files
bundle = OUTPUT / 'sentinel_fight_only_runtime.zip'
assert bundle.exists(), 'Chưa có gói runtime'
print('Experimental checkpoint, chưa được duyệt demo.')
files.download(str(bundle))
